# Flight Corridor AI — Data Pipeline

This notebook rebuilds Phase 1 & Phase 2 of the project cleanly:
1. Pull real PIREP (turbulence) data from AviationWeather.gov across 6 US regions
2. Clean, score, and deduplicate it
3. Save to Google Drive
4. Push it into Supabase (same database your n8n workflow now writes to)
5. Pull METAR wind data as a supporting feature

Run cells top to bottom. Each cell prints its own status so you can see exactly what happened.

## Step 0 — Setup

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

# AviationWeather.gov asks for a custom User-Agent
HEADERS = {"User-Agent": "student-project-flight-corridor-ai (contact: your_email@example.com)"}

PIREP_URL = "https://aviationweather.gov/api/data/pirep"
METAR_URL = "https://aviationweather.gov/api/data/metar"

print("Setup complete.")

Setup complete.


## Step 1 — Pull PIREPs across 6 US regions

**Important:** the bounding box format for this API is `bbox=minLat,minLon,maxLat,maxLon` (latitude first) — not the more common `minLon,minLat,maxLon,maxLat` GeoJSON order. Getting this backwards causes silent empty results (HTTP 204), not an error, so it's an easy trap.

In [ ]:
regions = {
    "NE":  "35,-90,50,-65",
    "SE":  "24,-90,35,-65",
    "MW":  "35,-105,50,-90",
    "SW":  "24,-105,35,-90",
    "NW":  "35,-125,50,-105",
    "SWC": "24,-125,35,-105",
}

all_pireps = []

for region_name, bbox in regions.items():
    params = {"format": "json", "age": 24, "bbox": bbox}
    resp = requests.get(PIREP_URL, headers=HEADERS, params=params)

    if resp.status_code == 200:
        data = resp.json()
        for record in data:
            record["region"] = region_name
        all_pireps.extend(data)
        print(f"{region_name}: {len(data)} PIREPs")
    elif resp.status_code == 204:
        print(f"{region_name}: no data (204)")
    else:
        print(f"{region_name}: error {resp.status_code} -> {resp.text[:200]}")

    time.sleep(1)  # respect the API's rate limit (max 100 req/min)

print("\nTotal PIREPs collected:", len(all_pireps))

NE: 294 PIREPs
SE: 400 PIREPs
MW: 400 PIREPs
SW: 240 PIREPs
NW: 313 PIREPs
SWC: 170 PIREPs

Total PIREPs collected: 1817


## Step 2 — Clean, score turbulence, and deduplicate

In [ ]:
raw_df = pd.DataFrame(all_pireps)
print("Raw shape:", raw_df.shape)

# Deduplicate reports that appear in more than one region (boundary overlap)
raw_df = raw_df.drop_duplicates(subset=["obsTime", "lat", "lon", "rawOb"]).reset_index(drop=True)
print("After dedup:", raw_df.shape)

Raw shape: (1817, 38)
After dedup: (1817, 38)


In [ ]:
# Map turbulence intensity text to a numeric ordinal risk score (0 = none, 8 = extreme)
turb_severity_map = {
    "": 0, "NEG": 0, "NEG-LGT": 1, "LGT": 2, "LGT-MOD": 3,
    "MOD": 4, "MOD-SEV": 5, "SEV": 6, "SEV-EXTM": 7, "EXTM": 8
}

def get_turb_score(row):
    s1 = turb_severity_map.get(row.get("tbInt1", "") or "", 0)
    s2 = turb_severity_map.get(row.get("tbInt2", "") or "", 0)
    return max(s1, s2)

raw_df["turb_score"] = raw_df.apply(get_turb_score, axis=1)
raw_df["obs_datetime"] = pd.to_datetime(raw_df["obsTime"], unit="s")

clean_df = raw_df[[
    "obs_datetime", "obsTime", "icaoId", "acType", "lat", "lon", "fltLvl",
    "tbInt1", "tbType1", "tbFreq1", "turb_score", "rawOb", "region"
]].copy().sort_values("obs_datetime").reset_index(drop=True)

print("Clean shape:", clean_df.shape)
print("\nTurbulence score distribution:")
print(clean_df["turb_score"].value_counts().sort_index())
print("\nTime range:", clean_df["obs_datetime"].min(), "to", clean_df["obs_datetime"].max())

clean_df.head(10)

Clean shape: (1817, 13)

Turbulence score distribution:
turb_score
0    1209
2     336
3      96
4     161
5       8
6       7
Name: count, dtype: int64

Time range: 2026-07-01 02:32:00 to 2026-07-02 02:23:00


,obs_datetime,obsTime,icaoId,acType,lat,lon,fltLvl,tbInt1,tbType1,tbFreq1,turb_score,rawOb,region
0,2026-07-01 02:32:00,1782873120,KWBC,SR22,32.8075,-116.8760,32,,,,0,SEE UA /OV SEE090005/TM 0232/FL032/TP SR22/SK ...,SWC
1,2026-07-01 02:32:00,1782873120,KWBC,A321,39.2340,-107.2332,350,LGT,,,2,ASE UA /OV DBL220020/TM 0232/FL350/TP A321/TB ...,NW
2,2026-07-01 02:33:00,1782873180,KWBC,B739,47.3356,-122.3096,58,,,,0,SEA UA /OV SEA161006/TM 0233/FL058/TP B739/SK ...,NW
3,2026-07-01 02:35:00,1782873300,KWBC,E75S,38.7391,-89.9186,100,LGT,CHOP,,2,BLV UA /OV TOY/TM 0235/FL100/TP E75S/TB LGT CH...,NE
4,2026-07-01 02:36:00,1782873360,KWBC,CL35,39.0739,-109.0061,400,LGT,CHOP,CONT,2,GJT UA /OV JNC260010/TM 0236/FL400/TP CL35/TB ...,NW
5,2026-07-01 02:37:00,1782873420,KWBC,E75L,43.8589,-116.3724,350,LGT,CHOP,OCNL,2,BOI UA /OV BOI320020/TM 0237/FL350/TP E75L/TB ...,NW
6,2026-07-01 02:38:00,1782873480,CYXU,DH8D,48.3719,-89.3239,0,,LLWS,,0,WG UUA /OV CYQT /TM 0238 /FLDURD /TP DH8D /RM ...,NE
7,2026-07-01 02:40:00,1782873600,KWBC,B38M,41.8495,-122.1803,380,LGT,,INT,2,MFR UA /OV OED120050/TM 0240/FL380/TP B38M/TB ...,NW
8,2026-07-01 02:50:00,1782874200,KWBC,B739,32.7399,-117.2285,33,,,,0,SAN UA /OV 2 W SAN/TM 0250/FL033/TP B739/SK BA...,SWC
9,2026-07-01 02:52:00,1782874320,KWBC,A312,33.8700,-116.4298,60,MOD,,,4,PSP UA /OV PSP/TM 0252/FL060/TP A312/TB MODERA...,SWC


## Step 3 — Save to Google Drive (survives Colab disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/flight_corridor_project'
os.makedirs(SAVE_DIR, exist_ok=True)

clean_df.to_csv(f'{SAVE_DIR}/pireps_clean.csv', index=False)
print(f"Saved {len(clean_df)} rows to {SAVE_DIR}/pireps_clean.csv")

Mounted at /content/drive
Saved 1817 rows to /content/drive/MyDrive/flight_corridor_project/pireps_clean.csv


## Step 4 — Push this batch into Supabase

This uses Supabase's REST API directly — no extra library needed, just `requests`. This lets Colab and n8n both write into the *same* table, so  dataset merges cleanly regardless of which one collected it.

**Fill in your own values below** (from Supabase → Settings → API):

In [ ]:
SUPABASE_URL = "............"   # <-- replace with your Project URL
SUPABASE_KEY = "..........."

supabase_headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json",
    "Prefer": "return=minimal"
}

# Build the row payload matching the `pireps` table columns we created in Supabase
def to_supabase_rows(df):
    rows = []
    for _, r in df.iterrows():
        rows.append({
            "obs_time": int(r["obsTime"]) if pd.notna(r["obsTime"]) else None,
            "icao_id": r["icaoId"],
            "ac_type": r["acType"],
            "lat": float(r["lat"]) if pd.notna(r["lat"]) else None,
            "lon": float(r["lon"]) if pd.notna(r["lon"]) else None,
            "flt_lvl": int(r["fltLvl"]) if pd.notna(r["fltLvl"]) else None,
            "tb_int1": r["tbInt1"],
            "tb_type1": r["tbType1"],
            "tb_freq1": r["tbFreq1"],
            "turb_score": int(r["turb_score"]),
            "raw_ob": r["rawOb"],
        })
    return rows

payload = to_supabase_rows(clean_df)

# Supabase REST insert has a practical batch size limit — send in chunks of 200
BATCH_SIZE = 200
inserted = 0

for i in range(0, len(payload), BATCH_SIZE):
    batch = payload[i:i+BATCH_SIZE]
    resp = requests.post(
        f"{SUPABASE_URL}/rest/v1/pireps",
        headers=supabase_headers,
        json=batch
    )
    if resp.status_code in (200, 201):
        inserted += len(batch)
        print(f"Inserted batch {i}-{i+len(batch)}: OK")
    else:
        print(f"Batch {i}-{i+len(batch)} FAILED: {resp.status_code} -> {resp.text[:300]}")

print(f"\nTotal inserted into Supabase: {inserted} / {len(payload)}")

Inserted batch 0-200: OK
Inserted batch 200-400: OK
Inserted batch 400-600: OK
Inserted batch 600-800: OK
Inserted batch 800-1000: OK
Inserted batch 1000-1200: OK
Inserted batch 1200-1400: OK
Inserted batch 1400-1600: OK
Inserted batch 1600-1800: OK
Inserted batch 1800-1817: OK

Total inserted into Supabase: 1817 / 1817


## Step 5 — Verify by reading it back from Supabase

In [ ]:
verify_resp = requests.get(
    f"{SUPABASE_URL}/rest/v1/pireps",
    headers=supabase_headers,
    params={"select": "*", "limit": "5", "order": "collected_at.desc"}
)

print("Status:", verify_resp.status_code)
verify_df = pd.DataFrame(verify_resp.json())
verify_df

Status: 200


,id,obs_time,icao_id,ac_type,lat,lon,flt_lvl,tb_int1,tb_type1,tb_freq1,turb_score,raw_ob,collected_at
0,1823,1782957120,KWBC,B739,48.7586,-109.9535,360,MOD,,,4,HVR UA /OV HVR315015/TM 0152/FL360/TP B739/TB ...,2026-07-02T02:47:04.345668+00:00
1,1824,1782957120,KWBC,A321,31.7108,-105.7080,340,LGT,CAT,CONT,2,ELP UA /OV ELP090030/TM 0152/FL340/TP A321/TB ...,2026-07-02T02:47:04.345668+00:00
2,1821,1782957060,KWBC,PC12,42.8329,-73.2891,150,MOD,CAT,,4,ALB UA /OV CAM180010/TM 0151/FL150/TP PC12/SK ...,2026-07-02T02:47:04.345668+00:00
3,1822,1782957120,KWBC,BE99,38.6359,-104.4739,100,MOD-SEV,,,5,COS UUA /OV BRK150020/TM 0152/FL100/TP BE99/TB...,2026-07-02T02:47:04.345668+00:00
4,1825,1782957120,KWBC,B737,33.9331,-118.4320,18,,,,0,LAX UA /OV LAX/TM 0152/FL018/TP B737/SK BKN015...,2026-07-02T02:47:04.345668+00:00


## Step 6 — Pull METAR wind data (supporting feature)

Handles missing columns gracefully (e.g. `wgst` — wind gust — is absent when there's no gusting wind, which caused a `KeyError` in the original version).

In [ ]:
airports = "KJFK,KORD,KDFW,KLAX,KATL"
metar_params = {"ids": airports, "format": "json"}

metar_resp = requests.get(METAR_URL, headers=HEADERS, params=metar_params)
print("Status:", metar_resp.status_code)

if metar_resp.status_code == 200 and metar_resp.text.strip():
    metars = metar_resp.json()
    metar_df = pd.DataFrame(metars)

    desired_cols = ["icaoId", "obsTime", "wspd", "wgst", "wdir"]
    available_cols = [c for c in desired_cols if c in metar_df.columns]
    missing_cols = [c for c in desired_cols if c not in metar_df.columns]

    if missing_cols:
        print("Note: these columns weren't present in this pull (normal — e.g. no gusts reported):", missing_cols)

    display_df = metar_df[available_cols] if not metar_df.empty else pd.DataFrame()
    metar_df.to_csv(f'{SAVE_DIR}/metars_raw.csv', index=False)
    print(f"Saved METARs to {SAVE_DIR}/metars_raw.csv")
else:
    print("No METAR data returned.")
    display_df = pd.DataFrame()

display_df

Status: 200
Note: these columns weren't present in this pull (normal — e.g. no gusts reported): ['wgst']
Saved METARs to /content/drive/MyDrive/flight_corridor_project/metars_raw.csv


,icaoId,obsTime,wspd,wdir
0,KLAX,1782959340,11,260
1,KDFW,1782957180,9,150
2,KATL,1782957120,6,300
3,KORD,1782957060,10,220
4,KJFK,1782957060,8,200


## Step 7 — Combined summary

Quick sanity check across everything pulled this run.

In [ ]:
print("=" * 50)
print("PIPELINE SUMMARY")
print("=" * 50)
print(f"PIREPs collected (this run):  {len(clean_df)}")
print(f"Turbulence score range:       {clean_df['turb_score'].min()} to {clean_df['turb_score'].max()}")
print(f"Time span (this run):         {clean_df['obs_datetime'].min()} -> {clean_df['obs_datetime'].max()}")
print(f"Rows now in Supabase 'pireps' table (running total, check dashboard for exact count)")
print(f"METAR stations pulled:        {airports}")
print("=" * 50)
print("\nNext: run this notebook again anytime to top up your dataset manually,")
print("or rely on the n8n workflow (writing to the same Supabase table every 30 min).")

PIPELINE SUMMARY
PIREPs collected (this run):  1817
Turbulence score range:       0 to 6
Time span (this run):         2026-07-01 02:32:00 -> 2026-07-02 02:23:00
Rows now in Supabase 'pireps' table (running total, check dashboard for exact count)
METAR stations pulled:        KJFK,KORD,KDFW,KLAX,KATL

Next: run this notebook again anytime to top up your dataset manually,
or rely on the n8n workflow (writing to the same Supabase table every 30 min).
